# 02 · Tutelas — Calidad y Normalización
**Objetivo:** Limpiar y validar el dataset para que el análisis del notebook 03 sea confiable.

Regla: **no se elimina ningún registro original**. Se agregan columnas de calidad y se documentan los problemas como hallazgos.

**Prerequisito:** haber corrido `01_tutelas_exploration.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_PATH = '../data/tutelas - Detalle1.csv'
REFERENCE_DATE = pd.Timestamp('2026-05-11')

## 1. Carga y parseo de fechas

In [2]:
df = pd.read_csv(DATA_PATH, encoding='utf-8', sep=None, engine='python')

date_cols = [
    'Fecha de creación',
    'Fecha y hora notificación tutela',
    'Fecha Vencimiento Admision',
    'Fecha Reasignacion Concepto Salud',
    'Fecha_Asignacion_Concepto',
    'Fecha Limite Concepto Admisión',
    'Fecha finalización concepto',
    'Fecha y hora de Contestación',
    'Fecha y hora notificación 1ra Instancia',
    'Fecha vencimiento fallo 1ra Instancia',
    'Fecha y hora Notificación 2da Instancia',
    'Fecha vencimiento fallo 2da Instancia',
    'Fecha y hora fallo Corte Constitucional',
    'Fecha vencimiento fallo corte',
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Registros cargados: {len(df):,}')

Registros cargados: 12,952


## 2. Normalización de texto en campos categóricos

In [3]:
# Estandarizar: mayúsculas, sin espacios extra
text_cols = [
    'Estado del ciclo de vida',
    'Compañía',
    'Regional T',
    'Area causal',
    'Motivo Causal Juridico',
    'Clasificación fallo 1ra Instancia',
    'Clasificación fallo 2da Instancia',
    'Tipos respuesta',
    'Medida provisional',
    'PBS/NO PBS',
    'Prestación sucesiva 1ra instancia',
    'Tipo de Prestación',
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace({'NAN': np.nan, 'NONE': np.nan})

print('Normalización de texto aplicada.')

Normalización de texto aplicada.


## 3. Detección de duplicados

In [4]:
# Duplicados por nombre de fichero (ID principal)
dup_fichero = df['Nombre del fichero'].duplicated(keep=False)
print(f'Registros con Nombre del fichero duplicado: {dup_fichero.sum()}')

# Duplicados exactos (todas las columnas)
dup_total = df.duplicated(keep=False)
print(f'Registros completamente duplicados: {dup_total.sum()}')

if dup_fichero.sum() > 0:
    print('\nEjemplos de ficheros duplicados:')
    print(df[dup_fichero][['Nombre del fichero', 'Estado del ciclo de vida', 'Fecha de creación']].head(6).to_string())

Registros con Nombre del fichero duplicado: 4
Registros completamente duplicados: 0

Ejemplos de ficheros duplicados:
                               Nombre del fichero Estado del ciclo de vida   Fecha de creación
1303     EXP-T-26-000001303 - 2026-0031 - 8276693                  NULIDAD 2026-01-16 08:03:58
2890  EXP-T-26-000002891 - 2026-0026 - 1040180343                  NULIDAD 2026-01-26 11:49:45
6032     EXP-T-26-000001303 - 2026-0031 - 8276693          ESPERA DE FALLO 2026-02-13 10:05:41
9664  EXP-T-26-000002891 - 2026-0026 - 1040180343     GESTIÓN FALLO 1 ÁREA 2026-03-09 10:42:59


## 4. Validación de coherencia temporal

In [5]:
inconsistencias = pd.DataFrame(index=df.index)

# Contestación antes de notificación (imposible)
inconsistencias['contestacion_antes_notif'] = (
    df['Fecha y hora de Contestación'] < df['Fecha y hora notificación tutela']
).fillna(False)

# Vencimiento admisión antes de notificación
inconsistencias['vencimiento_antes_notif'] = (
    df['Fecha Vencimiento Admision'] < df['Fecha y hora notificación tutela']
).fillna(False)

# Fallo 1ra instancia con fecha futura (posterior a fecha de análisis)
inconsistencias['fallo1_futuro'] = (
    df['Fecha vencimiento fallo 1ra Instancia'] > REFERENCE_DATE
).fillna(False)

# Contestación posterior al vencimiento (respuesta tardía — no inconsistencia, es un hallazgo)
inconsistencias['contestacion_tardia'] = (
    df['Fecha y hora de Contestación'] > df['Fecha Vencimiento Admision']
).fillna(False)

print('Resumen de validaciones temporales:')
for col in inconsistencias.columns:
    n = inconsistencias[col].sum()
    pct = n / len(df) * 100
    print(f'  {col}: {n:,} ({pct:.1f}%)')

Resumen de validaciones temporales:
  contestacion_antes_notif: 4 (0.0%)
  vencimiento_antes_notif: 14 (0.1%)
  fallo1_futuro: 6 (0.0%)
  contestacion_tardia: 1,150 (8.9%)


## 5. Columna de calidad de dato por registro

In [6]:
# Campos mínimos obligatorios para el análisis
campos_criticos = [
    'Fecha y hora notificación tutela',
    'Fecha Vencimiento Admision',
    'Regional T',
    'Area causal',
    'Estado del ciclo de vida',
]

nulos_criticos = df[campos_criticos].isnull().sum(axis=1)

df['calidad_dato'] = np.where(
    inconsistencias['contestacion_antes_notif'] | inconsistencias['vencimiento_antes_notif'],
    'Inconsistente',
    np.where(nulos_criticos > 0, 'Incompleto', 'OK')
)

print('Distribución calidad_dato:')
print(df['calidad_dato'].value_counts())
print(f'\n% registros confiables (OK): {(df["calidad_dato"] == "OK").mean()*100:.1f}%')

Distribución calidad_dato:
calidad_dato
Incompleto       7038
OK               5896
Inconsistente      18
Name: count, dtype: int64

% registros confiables (OK): 45.5%


## 6. Exportar dataset limpio

In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/tutelas_clean.csv', index=False, encoding='utf-8')
print(f'Guardado: ../data/processed/tutelas_clean.csv ({len(df):,} filas, {len(df.columns)} columnas)')
print('\nColumnas nuevas añadidas:')
nuevas = ['calidad_dato']
for c in nuevas:
    print(f'  + {c}')

Guardado: ../data/processed/tutelas_clean.csv (12,952 filas, 48 columnas)

Columnas nuevas añadidas:
  + calidad_dato


## 7. Resumen de hallazgos de calidad

**Anotar aquí para el documento ejecutivo:**

- % de registros en estado OK / Incompleto / Inconsistente
- Campos con mayor tasa de nulos críticos
- Inconsistencias temporales detectadas
- Riesgos de calidad por regional o área

> Continuar en `03_tutelas_analysis.ipynb`